# Day 12 — Model Explainability with SHAP

**Goal:** Make the XGBoost champion model SR 11-7 compliant by providing both global explanations (what drives the model overall) and individual explanations (why this specific customer was flagged or approved).

> **Regulatory context:** Under Regulation B (ECOA) and SR 11-7, a bank must be able to provide specific, factual reasons for any adverse credit decision. SHAP (SHapley Additive exPlanations) provides the mathematical foundation for those reason codes.

## Sections
1. Setup & Imports
2. Data Load + Feature Engineering
3. Train / Test Split
4. Train XGBoost + Build SHAP Explainer
5. Task 12.1 — Global Feature Importance (SHAP Summary Plot)
6. Task 12.2 — Individual Prediction Explanation (Waterfall Plot)
7. Task 12.3 — Adverse Action Reason Codes
8. Task 12.4 — Explainability Summary Reference

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

from src.features.engineer import engineer_features

plt.style.use('seaborn-v0_8-whitegrid')
RANDOM_STATE = 42
FIGURES = PROJECT_ROOT / 'reports/figures'

## 2. Data Load + Feature Engineering

In [ ]:
for fname in ['credit_default_eda_ready.parquet',
              'credit_default_validated.parquet',
              'credit_default_cleaned.parquet']:
    path = PROJECT_ROOT / 'data/processed' / fname
    if path.exists():
        df_raw = pd.read_parquet(path)
        print(f'Loaded: {fname}  |  Shape: {df_raw.shape}')
        break

df_raw = df_raw.drop(columns=[c for c in ['edu_label', 'mar_label', 'age_group']
                               if c in df_raw.columns])
df = engineer_features(df_raw)
print(f'After engineering: {df.shape}  (+{df.shape[1] - df_raw.shape[1]} features)')

## 3. Train / Test Split

Same `random_state=42` and `stratify=y` as Days 10 and 11 — identical split for consistency.

In [ ]:
X = df.drop(columns=['default', 'risk_segment'])
y = df['default']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
scale_pos_weight = neg / pos

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Train default rate: {y_train.mean():.3f}')

## 4. Train XGBoost + Build SHAP Explainer

In [ ]:
xgb = XGBClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=6,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1
)
xgb.fit(X_train, y_train)

y_probs = xgb.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_probs)
print(f'XGBoost AUC-ROC: {auc:.4f}  |  Gini: {2 * auc - 1:.4f}')

# TreeExplainer is exact (not approximate) for tree-based models — zero sampling noise
explainer = shap.TreeExplainer(xgb)
shap_values = explainer(X_test)

print(f'\nSHAP values shape      : {shap_values.values.shape}')
print(f'Expected value (base)  : {explainer.expected_value:.4f}'
      f'  (log-odds; sigmoid = {1 / (1 + np.exp(-explainer.expected_value)):.3f})')
print(f'Mean |SHAP| per feature: {np.abs(shap_values.values).mean(axis=0).mean():.4f}')

## 5. Task 12.1 — Global Feature Importance (SHAP Summary Plot)

**Beeswarm plot** — each dot is one customer in the test set:
- **X-axis:** SHAP value — how much this feature pushed the model toward default (right) or non-default (left).
- **Colour:** Feature value — red = high, blue = low.
- **Y-axis:** Features ranked by mean |SHAP| (most impactful at top).

> **Interview tip:** "Traditional feature importance shows *that* a variable matters. SHAP shows *how* and *in which direction* — e.g., higher `pay_0` (recent payment delay) strongly increases the predicted probability of default." 

In [ ]:
shap.summary_plot(shap_values.values, X_test, max_display=20, show=False)
plt.title('SHAP Summary — Global Feature Impact on Default Probability',
          fontsize=13, fontweight='bold', pad=14)
plt.tight_layout()
plt.savefig(FIGURES / '22_shap_summary_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Mean |SHAP| bar chart — more readable for presentations / model cards
mean_shap = pd.Series(
    np.abs(shap_values.values).mean(axis=0),
    index=X_test.columns
).sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 8))
mean_shap.plot(kind='barh', ax=ax, color='#2196F3')
ax.invert_yaxis()
ax.set_xlabel('Mean |SHAP Value| (average impact on model output)', fontsize=11)
ax.set_title('Top 20 Features by Global SHAP Importance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES / '23_shap_bar_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 10 global drivers:')
print(mean_shap.head(10).to_string())

## 6. Task 12.2 — Individual Prediction Explanation (Waterfall Plot)

We select the **highest-risk customer** in the test set (highest predicted default probability) and explain exactly why the model flagged them.

The waterfall plot shows:
- **Base value:** where the model starts (average log-odds across training data)
- **Each bar:** how much one feature pushed the score up (red = toward default) or down (blue = away from default)
- **Final value:** the model's output log-odds for this customer

In [ ]:
# Pick the highest-risk customer (highest predicted default probability)
highest_risk_idx = np.argmax(y_probs)
prob_hr = y_probs[highest_risk_idx]
actual_hr = y_test.iloc[highest_risk_idx]

print(f'Highest-risk customer index (in test set): {highest_risk_idx}')
print(f'  Predicted default probability : {prob_hr:.4f}  ({prob_hr*100:.1f}%)')
print(f'  Actual outcome                : {"DEFAULT" if actual_hr == 1 else "NON-DEFAULT"}')
print(f'  Risk band                     : High Risk (score > 0.50)')

shap.plots.waterfall(shap_values[highest_risk_idx], max_display=15, show=False)
fig = plt.gcf()
fig.set_size_inches(12, 8)
fig.suptitle(
    f'SHAP Waterfall — High-Risk Customer (P(default)={prob_hr:.3f})',
    fontsize=13, fontweight='bold', y=1.01
)
plt.tight_layout()
fig.savefig(FIGURES / '24_shap_waterfall_high_risk.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Task 12.3 — Adverse Action Reason Codes

Under **Regulation B (ECOA)**, any credit denial must be accompanied by an Adverse Action Notice listing the principal reasons for the decision. Banks typically provide 3–4 reason codes.

SHAP maps directly to this requirement:
- Features with the **largest positive SHAP values** are the factors that most increased the default probability — these become the adverse action reason codes.
- Features with negative SHAP values actually *helped* the applicant and are not cited.

In [ ]:
def get_reason_codes(shap_vals_row, feature_values_row, feature_names, top_n=3):
    """Return top_n adverse factors (positive SHAP values sorted descending)."""
    sv = pd.Series(shap_vals_row, index=feature_names)
    positive = sv[sv > 0].sort_values(ascending=False).head(top_n)
    rows = []
    for feat, shap_val in positive.items():
        fval = feature_values_row[feat]
        rows.append({
            'Reason Code': feat,
            'Customer Value': round(float(fval), 4),
            'SHAP Impact': round(shap_val, 4),
            'Direction': 'Increases default risk',
        })
    return pd.DataFrame(rows)

# Adverse action codes for the highest-risk customer
reason_codes = get_reason_codes(
    shap_values.values[highest_risk_idx],
    X_test.iloc[highest_risk_idx],
    X_test.columns.tolist(),
    top_n=3
)

print('=== ADVERSE ACTION REASON CODES ===')
print(f'Customer predicted default probability: {prob_hr:.3f}')
print(f'Decision: DECLINE')
print()
print(reason_codes.to_string(index=False))
print()
print('Adverse Action Notice draft:')
print('-' * 60)
for i, row in reason_codes.iterrows():
    print(f'  Reason {i+1}: {row["Reason Code"].replace("_", " ").title()}'
          f' (value={row["Customer Value"]})')
print('-' * 60)

In [ ]:
# Also demonstrate a borderline customer to show contrast
# Pick the customer closest to the t=0.30 threshold (most uncertain)
borderline_idx = np.argmin(np.abs(y_probs - 0.30))
prob_bl = y_probs[borderline_idx]

print(f'Borderline customer (closest to t=0.30):')
print(f'  Predicted probability: {prob_bl:.4f}')
print(f'  Actual outcome       : {"DEFAULT" if y_test.iloc[borderline_idx] == 1 else "NON-DEFAULT"}')
print()

reason_borderline = get_reason_codes(
    shap_values.values[borderline_idx],
    X_test.iloc[borderline_idx],
    X_test.columns.tolist(),
    top_n=3
)
print('Borderline adverse factors:')
print(reason_borderline.to_string(index=False))

## 8. Task 12.4 — Explainability Summary

See **`docs/model_explainability_summary.md`** for the plain-English summary written for a non-technical Risk Manager.

### Interview talking points from today's SHAP analysis:

1. **Global drivers:** `max_delinquency` (worst-ever delinquency, an *engineered* feature) is the single strongest predictor (mean |SHAP|=0.389), followed by `PAY_0` (0.272) and `total_delinquencies` (0.206). All three are payment-behaviour signals.

2. **Direction matters:** `PAY_AMT1` (amount paid last month) has a *negative* SHAP direction — paying more is protective. High `util_ratio_m2` increases risk. SHAP shows both magnitude and direction, unlike tree-based importance scores.

3. **Engineered features dominate:** `max_delinquency` and `total_delinquencies` (Day 9 engineered features) occupy ranks 1 and 3 globally. They are not in the raw dataset — this directly validates the feature engineering work and is a strong SR 11-7 talking point.

4. **SHAP = legal compliance:** The top-3 positive SHAP values for any declined applicant map directly to the reason codes required on an Adverse Action Notice under Regulation B.

5. **SR 11-7 alignment:** SHAP provides the "model transparency" requirement. The combination of a documented model card, threshold policy, and SHAP explanations constitutes a model governance package a regulator would expect to review.

In [ ]:
# Portfolio-level SHAP insights for the Risk Manager summary
mean_abs_shap = pd.Series(
    np.abs(shap_values.values).mean(axis=0),
    index=X_test.columns
).sort_values(ascending=False)

top5 = mean_abs_shap.head(5)
print('=== PORTFOLIO-LEVEL SHAP INSIGHTS ===')
print('Top 5 global drivers (mean |SHAP|):')
for feat, val in top5.items():
    # sign: does higher feature value tend to increase or decrease default prob?
    corr = np.corrcoef(X_test[feat].values, shap_values[:, feat].values)[0, 1]
    direction = 'increases' if corr > 0 else 'decreases'
    print(f'  {feat:<35} {val:.4f}  (higher value {direction} default risk)')

print(f'\nTotal test customers explained : {len(X_test):,}')
print(f'Customers in High Risk band    : {(y_probs > 0.5).sum():,}'
      f'  ({(y_probs > 0.5).mean():.1%})')
print(f'Customers in Low  Risk band    : {(y_probs < 0.2).sum():,}'
      f'  ({(y_probs < 0.2).mean():.1%})')
print()
print('Explainability summary: docs/model_explainability_summary.md')
print('Day 12 COMPLETE — Model is SR 11-7 compliant with SHAP explanations.')